In [1]:
using Pkg
Pkg.activate("C:/Users/ibzja/Documents/UPF_2022_2026/4t/2n_trimestre/Practiques_tutelades/CellBasedModels.jl")
using CellBasedModels 
using GeometryBasics
using Distributions
using GLMakie, Colors
Makie.inline!(true)
using CSV, DataFrames, Statistics
using Printf, JLD2
using SpecialFunctions
using LsqFit
using LinearAlgebra
using DifferentialEquations, StaticArrays

  Activating project at `C:\Users\ibzja\Documents\UPF_2022_2026\4t\2n_trimestre\Practiques_tutelades\CellBasedModels.jl`


In [27]:
cripts = ABM(2,
    agent = Dict(
        :vx => Float64,
        :vy => Float64,
        :v => Float64,  #Swimming speed
        :theta => Float64,
        :d => Float64,
        :l => Float64,
        :m => Float64,
        :fx => Float64,
        :fy => Float64,
        :W => Float64,
        :pressure => Float64,
        :active => Bool,

        :S => Float64,
        # :S_2 => Float64,

        :methyl => Float64, #Receptor methylation
        :Yp => Float64, #CheYP levels, probability of tumblingç
        :G => Float64,
        :λ => Float64,
        :P => Float64,
        :M => Float64,
        :F => Float64,
        :A => Float64,
        :M => Float64,
        :Ds => Float64
    ),

    model = Dict(

        :Dr_run => Float64,

        :ε0 => Float64, #Energy parameters
        :ε1 => Float64,
        :ε2 => Float64,
        :ε3 => Float64,
        :K => Float64,
        :Nrec => Float64, #Cooperativity
        :Ki => Float64, #Dissociation constants
        :Ka => Float64,
        :τm => Float64, #Methylation adaptation timescale
        :α => Float64,      #Total Yp pool
        :ωFrec => Float64,     #Basal switching frequency
        :Ky => Float64,         #CheA - CheY phosphorylation rate
        :Z => Float64,          #CheZ concentration
        :Kz => Float64,         #CheZ mediated dephosphorylation rate
        :Yy => Float64,         #Basa Yp leak

        :DMedium => Float64,
        :delta => Float64
    ),

    medium = Dict(
        :mm => Float64,
        :ve => Float64
    ),

    agentODE = quote
  
        xmin, xmax = simBox[1,1], simBox[1,2]
        ymin, ymax = simBox[2,1], simBox[2,2]


        idx = Int(floor(Int, x/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)
        idy = Int(floor(Int, y/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)
        
        # X direction
        if x < xmin
            idx = Int(floor(Int, (x+(xmax - xmin))/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)

        elseif x > xmax
            idx = Int(floor(Int, (x-(xmax - xmin))/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)

        end

        # # Y direction
        if y < ymin
            idy = Int(floor(Int,(y+ymax-ymin)/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)

        elseif y > ymax
            idy = Int(floor(Int,(y-(ymax-ymin))/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)

        end

        mmb = max(0, mm[idx,idy])

        F = ε0 + ε1 * methyl + Nrec * log((1 + mmb / Ki) / (1 + mmb / Ka))
        F0 = log(((Ky * (α - K)) / (K * (Kz * Z + Yy))) - 1)      

        mx = (ε0 + Nrec * log((1 + mmb / Ki) / (1 + mmb / Ka)) - F0) / (- ε1)
       

        A = 1 / (1 + exp(F))    

        Yp = (Ky * A * α) / ((Ky * A) + (Kz * Z) + Yy)

        G = ε2 / 4 - (ε3 / 2) / (1 + (K / Yp))     

        dt(x) = vx 
        dt(y) = vy  
        dt(methyl) = -(1 / τm) * (methyl - mx)     
        
    end,

    agentRule = quote

        xmin, xmax = simBox[1,1], simBox[1,2]
        ymin, ymax = simBox[2,1], simBox[2,2]

        idx = Int(floor(Int, x/(simBox[1,2]-simBox[1,1])*NMedium[1])+NMedium[1]/2)
        idy = Int(floor(Int, y/(simBox[2,2]-simBox[2,1])*NMedium[2])+NMedium[2]/2)        
       
     # Adding population every 100 steps
        p_add = 0.001

        if i1_ == 1
            if rand() < p_add
                for k in 1:10
                    @addAgent(
                        x = xmin + 10,
                        y = ymax/2 + 10 + rand()*(ymax - (ymax/2) - 10),
                        theta = 2 * pi,
                        l = 3
                    )
                end
            end
        end

        v_run = v
        v_tumble = 0.25 
        speed = active ? v_run : v_tumble

        Dr_tumble = 6.2      
        Dr_total = active ? Dr_run : Dr_tumble

        mm[idx,idy] += S

        if active 
            λ = ωFrec*exp(-G)
            P = 1 - exp(-λ * dt)
                
        else
            λ = ωFrec*exp(G)
            P = 1 - exp(-λ * dt)  
                
        end


        if active 
            λrt = ωFrec*exp(-G) 
            P_rt = 1 - exp(-λrt * dt)
            P = rand() 
                                                  
            if P < P_rt             
                active = false
                vx = speed* cos(theta) + ve[idx, idy]*0.5
                vy = speed* sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn() 
                        
            else    
                active = true
                vx = speed * cos(theta) + ve[idx, idy]*0.5
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()
                     
            end

        else
            λtr = ωFrec*exp(G) 
            P_tr = 1 - exp(-λtr * dt)
            P = rand()

            if P < P_tr
                active = true
                vx = speed * cos(theta) + ve[idx, idy]*0.5
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()

            else
                active = false
                vx = speed* cos(theta) + ve[idx, idy]*0.5
                vy = speed* sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()

            end
        end

        if x <= xmin
            x = xmin + (xmin - x)
            theta = pi - theta
        elseif x >= xmax
            @removeAgent()
        end

        if y >= ymax
            y = 2*ymax - y
            theta = 2*pi - theta
        end

        if y <= wall_y(x, 62, depth, 250, 20, 500)
            x_new = clamp(round(Int, x/2), 1, 250)
            y_new = clamp(round(Int, y/2), 1, 250)

            if MYT[x_new, y_new] == 1 && MXR[x_new, y_new] == 1
                y = wall_y(x, 62, depth, 250, 20, 500) + 1
                x = x + 1
                theta = theta + pi

            elseif MYT[x_new, y_new] == 1 && MXL[x_new, y_new] == 1
                y = wall_y(x, 62, depth, 250, 20, 500) + 1
                x = x - 1
                theta = theta + pi

            elseif MYT[x_new, y_new] == 1 
                y = wall_y(x, 62, depth, 250, 20, 500) + 1
                theta = 2*pi - theta

            elseif MXR[x_new, y_new] == 1
                x = x + 2
                theta = pi - theta

            elseif MXL[x_new, y_new] == 1
                x = x - 2
                theta = pi - theta

            elseif M0[x_new, y_new] == 1        # Esta dins, no pinta res allí
                t0 = round(Int, (t-1))
                x_old = x[t0]
                y_old = y[t0]
                nsteps = ceil(Int, max(abs(x - x_old), abs(y - y_old)))

                hit_type = nothing

                for s in 0:nsteps
                    xs = x_old + (x - x_old) * s / nsteps
                    ys = y_old + (y - y_old) * s / nsteps

                    xi = clamp(round(Int, xs/2), 1, 250)
                    yi = clamp(round(Int, ys/2), 1, 250)

                    if MXR[xi, yi]
                        hit_type = :right
                        break
                    elseif MXL[xi, yi]
                        hit_type = :left
                        break
                    elseif MYT[xi, yi]
                        hit_type = :horizontal
                        break
                    end
                end
                if hit_type == :right
                    x = x_old
                    theta = pi - theta

                elseif hit_type == :left
                    x = x_old
                    theta = pi - theta

                elseif hit_type == :horizontal
                    y = y_old
                    theta = 2*pi - theta
                end
                                
            end
        end
    end,


    mediumODE = quote

        if @mediumInside()
    # --- geometry mask FIRST (do NOT overwrite mm itself) ---
            if M0[i1_, i2_]
                dt(mm) = 0

            else
                dt(mm) = DMedium * (@∂2(1, mm) + @∂2(2, mm)) - delta * mm - ve[i1_, i2_] * @∂(1, mm)

            end
            mm = MXR[i1_, i2_] ? mm[i1_ + 1, i2_] : mm
            mm = MXL[i1_, i2_] ? mm[i1_ - 1, i2_] : mm
            mm = MYT[i1_, i2_] ? mm[i1_, i2_ + 1] : mm
            
        elseif @mediumBorder(1,-1) # left PBC
            mm = mm[1, i2_]

        elseif @mediumBorder(1,1) #right PBC
            mm = mm[NMedium[1] - 1, i2_]

        elseif @mediumBorder(2,-1) # down NBC (newmann)
            mm = 0

        elseif @mediumBorder(2,+1)
            mm = mm[i1_, NMedium[2] - 1]

        end
    end,


    agentAlg = CBMIntegrators.Heun(),
    mediumAlg=DifferentialEquations.Euler()
)

PARAMETERS
	x (Float64 agent)
	y (Float64 agent)
	xₘ (Float64 medium)
	yₘ (Float64 medium)
	Ds (Float64 agent)
	F (Float64 agent)
	active (Bool agent)
	methyl (Float64 agent)
	l (Float64 agent)
	S (Float64 agent)
	M (Float64 agent)
	d (Float64 agent)
	λ (Float64 agent)
	v (Float64 agent)
	A (Float64 agent)
	fx (Float64 agent)
	vx (Float64 agent)
	fy (Float64 agent)
	m (Float64 agent)
	Yp (Float64 agent)
	P (Float64 agent)
	pressure (Float64 agent)
	vy (Float64 agent)
	W (Float64 agent)
	G (Float64 agent)
	theta (Float64 agent)
	ε1 (Float64 model)
	α (Float64 model)
	Z (Float64 model)
	Dr_run (Float64 model)
	Ka (Float64 model)
	ε3 (Float64 model)
	ε0 (Float64 model)
	DMedium (Float64 model)
	delta (Float64 model)
	Ky (Float64 model)
	Kz (Float64 model)
	K (Float64 model)
	ε2 (Float64 model)
	Nrec (Float64 model)
	τm (Float64 model)
	Yy (Float64 model)
	Ki (Float64 model)
	ωFrec (Float64 model)
	mm (Float64 medium)
	ve (Float64 medium)


UPDATE RULES
mediumODE
 if @mediumInside()
    if

In [19]:
function wall_y(x, w, d, y0, xstart, xend)
    
    # outside wall region → flat at baseline
    if x < xstart || x > xend
        return y0
    end

    # shift x so pattern starts at xstart
    ξ = x - xstart

    # square-wave pattern (0 or -d), then shift up by y0
    return y0 - d * floor((1 + sign(sin(2π * ξ / w))) / 2)
end

wall_y (generic function with 1 method)

In [21]:
        x1, x2, y1, y2 = 0, 500, 0, 500
        med1, med2 = 250, 250

        width = 62
        depth = 100
        y_start = y2/2
        x_start = 20
        x_end = x2
        dx = (x2 - x1) / med1
        dy = (y2 -y1) / med2

        xcoord(i1_) = x1 + (i1_ - 1) * dx
        ycoord(i2_) = y1 + (i2_ - 1) * dy
        Nx = med1
        Ny = med2

        mask_raw = zeros(Bool, med1, med2)
        M0 = zeros(Bool, med1, med2)
        MXL = zeros(Bool, med1, med2)
        MXR = zeros(Bool, med1, med2)
        MYT = zeros(Bool, med1, med2)

        for i in 1:Nx, j in 1:Ny
            x = xcoord(i)
            y = ycoord(j)
            mask_raw[i,j] = y < wall_y(x, width, depth, y_start, x_start, x_end)
        end

        for i in 2:Nx-1, j in 2:Ny-1
            if mask_raw[i,j] &&
            mask_raw[i+1,j] &&
            mask_raw[i-1,j] &&
            mask_raw[i,j+1] &&
            mask_raw[i,j-1] &&
            mask_raw[i+1,j+1] &&
            mask_raw[i+1,j-1] &&
            mask_raw[i-1,j+1] &&
            mask_raw[i-1,j-1]

                M0[i,j] = true
            end
        end

        for i in 2:Nx-1, j in 2:Ny-1

            if M0[i,j] == 1

                # --- HORIZONTAL LINES ---
                # bottom edge (neighbor below is outside)
                if M0[i, j-1] == 0
                    MYT[i,j] = true
                end

                # top edge (neighbor above is outside)
                if M0[i, j+1] == 0
                    MYT[i,j] = true
                end


                # --- LEFT WALL ---
                if M0[i-1, j] == 0
                    MXL[i,j] = true
                end


                # --- RIGHT WALL ---
                if M0[i+1, j] == 0
                    MXR[i,j] = true
                end

            end
        end

In [28]:
Dc = 10
delta = 0.0025
depths = [100, 150, 200]
ve = 10
ns = [0.5, 2.0, 5.0, 10.0]

for (idx, n) in enumerate(ns)

    for i in 1:length(depths)

        println("Running simulation with parameters: ", depths[i], " and ", n)

        x1, x2, y1, y2 = 0, 500, 0, 500
        med1, med2 = 250, 250

        width = 62
        depth = depths[i]
        y_start = y2/2
        x_start = 20
        x_end = x2
        dx = (x2 - x1) / med1
        dy = (y2 -y1) / med2

        xcoord(i1_) = x1 + (i1_ - 1) * dx
        ycoord(i2_) = y1 + (i2_ - 1) * dy
        Nx = med1
        Ny = med2

        mask_raw = zeros(Bool, med1, med2)
        M0 = zeros(Bool, med1, med2)
        MXL = zeros(Bool, med1, med2)
        MXR = zeros(Bool, med1, med2)
        MYT = zeros(Bool, med1, med2)

        for i in 1:Nx, j in 1:Ny
            x = xcoord(i)
            y = ycoord(j)
            mask_raw[i,j] = y < wall_y(x, width, depth, y_start, x_start, x_end)
        end

        for i in 2:Nx-1, j in 2:Ny-1
            if mask_raw[i,j] &&
            mask_raw[i+1,j] &&
            mask_raw[i-1,j] &&
            mask_raw[i,j+1] &&
            mask_raw[i,j-1] &&
            mask_raw[i+1,j+1] &&
            mask_raw[i+1,j-1] &&
            mask_raw[i-1,j+1] &&
            mask_raw[i-1,j-1]

                M0[i,j] = true
            end
        end

        for i in 2:Nx-1, j in 2:Ny-1

            if M0[i,j] == 1

                # --- HORIZONTAL LINES ---
                # bottom edge (neighbor below is outside)
                if M0[i, j-1] == 0
                    MYT[i,j] = true
                end

                # top edge (neighbor above is outside)
                if M0[i, j+1] == 0
                    MYT[i,j] = true
                end


                # --- LEFT WALL ---
                if M0[i-1, j] == 0
                    MXL[i,j] = true
                end


                # --- RIGHT WALL ---
                if M0[i+1, j] == 0
                    MXR[i,j] = true
                end

            end
        end

        com = Community(
            cripts,
            N=1,
            dt=0.01,
            simBox = [x1 x2; y1 y2],
            NMedium = [med1, med2]
        )

        m = 1/100
        g = 1/10000
        d = 1

        com.Dr_run = 0.062

        com.v = 20.0    #Velocitat bacteries biològica

        com.ωFrec = 1.3
        com.Ki = 0.0182
        com.Ka = 3.0
        com.Nrec = 6.0
        com.ε0   = 6.0
        com.ε1   = -1.0
        com.ε2   = 80
        com.ε3   = 80

        com.τm = 1

        com.α   = 6.0
        com.K = 2.0 
        com.Ky = 100.0
        com.Kz = 10.0
        com.Z = 5.0
        com.Yy = 0.

        com.m = 1.        
        com.d = 1.        
        com.l = 3;

        com.DMedium = Dc / n
        com.delta = delta * n

        com.x = 10
        com.y = 400
        com.theta = rand(Uniform(pi/2,(3*pi)/2),com.N)

        com.active .= true 
        com.methyl .= 0.0
        com.Yp .= com.K
        com.S = 0.0025

        com.ve = zeros(com.NMedium[1], com.NMedium[2])

        for i1 in 1:com.NMedium[1], i2 in 1:com.NMedium[2]
            y = ycoord(i2)

            if y > (y2 / 2)
                com.ve[i1, i2] = ve
            else
                com.ve[i1, i2] = 0.0
            end
        end

        outfile = "c_n$(n)_lc_$(depths[i]).jld2"
        steps = 100000

        loadToPlatform!(com, preallocateAgents=10000)
        com.mm = zeros(Float64, com.NMedium...)

        jldopen(outfile, "w") do file
            
            for step in 1:steps
                CellBasedModels.step!(com)
                if step % 100 == 0
                    stepname = @sprintf("step_%06d", step)
                    g = JLD2.Group(file, stepname)

                    # Agent-level arrays (length = N)
                    g["x"] = copy(com.x)
                    g["y"] = copy(com.y)
                    g["theta"] = copy(com.theta)

                    # Medium grid (saved once per step)
                    g["mm_grid"] = copy(com.mm)
                end
            end
        end
    end
end


Running simulation with parameters: 100 and 0.5
Running simulation with parameters: 150 and 0.5
Running simulation with parameters: 200 and 0.5
Running simulation with parameters: 100 and 2.0
Running simulation with parameters: 150 and 2.0
Running simulation with parameters: 200 and 2.0
Running simulation with parameters: 100 and 5.0
Running simulation with parameters: 150 and 5.0
Running simulation with parameters: 200 and 5.0
Running simulation with parameters: 100 and 10.0
Running simulation with parameters: 150 and 10.0
Running simulation with parameters: 200 and 10.0


In [34]:
steps = 100000
steps_1 = 100:100:steps
sizes = length(steps_1)

1000

In [ ]:
Dc = 10
delta = 0.0025
depths = [100, 150, 200]
ve = 10
ns = [0.5, 2.0, 5.0, 10.0]
steps = 100000

for (idx, n) in enumerate(ns)

    for i in 1:length(depths)

        println("Running simulation with parameters: ", depths[i], " and ", n)

        x1, x2, y1, y2 = 0, 500, 0, 500
        med1, med2 = 250, 250

        width = 62
        depth = depths[i]
        y_start = y2/2
        x_start = 20
        x_end = x2
        dx = (x2 - x1) / med1
        dy = (y2 -y1) / med2

        xcoord(i1_) = x1 + (i1_ - 1) * dx
        ycoord(i2_) = y1 + (i2_ - 1) * dy
        Nx = med1
        Ny = med2

        mask_raw = zeros(Bool, med1, med2)
        M0 = zeros(Bool, med1, med2)
        MXL = zeros(Bool, med1, med2)
        MXR = zeros(Bool, med1, med2)
        MYT = zeros(Bool, med1, med2)

        for i in 1:Nx, j in 1:Ny
            x = xcoord(i)
            y = ycoord(j)
            mask_raw[i,j] = y < wall_y(x, width, depth, y_start, x_start, x_end)
        end

        for i in 2:Nx-1, j in 2:Ny-1
            if mask_raw[i,j] &&
            mask_raw[i+1,j] &&
            mask_raw[i-1,j] &&
            mask_raw[i,j+1] &&
            mask_raw[i,j-1] &&
            mask_raw[i+1,j+1] &&
            mask_raw[i+1,j-1] &&
            mask_raw[i-1,j+1] &&
            mask_raw[i-1,j-1]

                M0[i,j] = true
            end
        end

        for i in 2:Nx-1, j in 2:Ny-1

            if M0[i,j] == 1

                # --- HORIZONTAL LINES ---
                # bottom edge (neighbor below is outside)
                if M0[i, j-1] == 0
                    MYT[i,j] = true
                end

                # top edge (neighbor above is outside)
                if M0[i, j+1] == 0
                    MYT[i,j] = true
                end


                # --- LEFT WALL ---
                if M0[i-1, j] == 0
                    MXL[i,j] = true
                end


                # --- RIGHT WALL ---
                if M0[i+1, j] == 0
                    MXR[i,j] = true
                end

            end
        end

        data = Dict{Int, Any}()
        jldopen("c_n$(n)_lc_$(depths[i]).jld2", "r") do file
            for step in 100:100:steps
                key = file[@sprintf("step_%06d", step)]
                data[step] = Dict(
                    "x" => copy(key["x"]),
                    "y" => copy(key["y"]),
                    "mm_grid" => copy(key["mm_grid"])
                )
            end
        end

        # Fer matrius per cada set
        ncripts = 8
        right_wall = findall(MXR[:, 100] .== 1) .* 2
        left_wall = findall(MXL[:, 100] .== 1) .* 2
        x1_1, x1_2, x1_3, x1_4, x1_5, x1_6, x1_7, x1_8 = right_wall[1:8]
        x2_1, x2_2, x2_3, x2_4, x2_5, x2_6, x2_7, x2_8 = left_wall[2:9]

        y_bottom = y_start - depth
        y_top = y_start

        counts = zeros(ncripts, steps)
        steps_1 = 100:100:steps
        sizes = length(steps_1)

        for (idx, i) in enumerate(steps_1)
            g = data[i]
            x = g["x"]
            y = g["y"]

            count1 = 0
            count2 = 0
            count3 = 0
            count4 = 0
            count5 = 0
            count6 = 0
            count7 = 0
            count8 = 0

            for j in 1:length(x)
            
                if x2_1 > x[j] > x1_1 && y[j] < y_top
                    count1 += 1
                elseif x2_2 > x[j] > x1_2 && y[j] < y_top
                    count2 += 1
                elseif x2_3 > x[j] > x1_3 && y[j] < y_top
                    count3 += 1
                elseif x2_4 > x[j] > x1_4 && y[j] < y_top
                    count4 += 1
                elseif x2_5 > x[j] > x1_5 && y[j] < y_top
                    count5 += 1
                elseif x2_6 > x[j] > x1_6 && y[j] < y_top
                    count6 += 1
                elseif x2_7 > x[j] > x1_7 && y[j] < y_top
                    count7 += 1
                elseif x2_8 > x[j] > x1_8 && y[j] < y_top
                    count8 += 1
                end
            end

            counts[1, i] = count1
            counts[2, i] = count2
            counts[3, i] = count3
            counts[4, i] = count4
            counts[5, i] = count5
            counts[6, i] = count6
            counts[7, i] = count7
            counts[8, i] = count8
        end

        fig = Figure(size=(900, 600))
        ax = Axis(fig[1, 1], xlabel = "Time", ylabel = "Nº cells", title = "Cript colonization over time")
        lines!(ax, (1:steps)*0.01, counts[1, :], label = "Cript 1", linewidth = 1)
        lines!(ax, (1:steps)*0.01, counts[2, :], label = "Cript 2", linewidth = 1)
        lines!(ax, (1:steps)*0.01, counts[3, :], label = "Cript 3", linewidth = 1)
        lines!(ax, (1:steps)*0.01, counts[4, :], label = "Cript 4", linewidth = 1)
        lines!(ax, (1:steps)*0.01, counts[5, :], label = "Cript 5", linewidth = 1)
        lines!(ax, (1:steps)*0.01, counts[6, :], label = "Cript 6", linewidth = 1)
        lines!(ax, (1:steps)*0.01, counts[7, :], label = "Cript 7", linewidth = 1)
        lines!(ax, (1:steps)*0.01, counts[8, :], label = "Cript 8", linewidth = 1)

        fig[1,2] = Legend(fig, ax, "Cripts")

        save("Plots/Cripts/Test_sims/colonization_time_n$(n)_lc$(depths[i]).png", fig)


        grid_values = zeros(ncripts, steps)

        x2 = Int.(left_wall[2:9] ./ 2)
        x1 = Int.(right_wall[1:8] ./2)
        y1 = Int(y_bottom / 2)
        y2 = Int(y_start / 2)

        for (idx, i) in enumerate(steps_1)
            g = data[i]
            grid = g["mm_grid"]
            for j in 1:ncripts
            
                cript = grid[x1[j]:x2[j], y1:y2]
                mm_mean = mean(cript)

                grid_values[j, i] = mm_mean

            end
        end

        fig2 = Figure(size=(900, 600))
        ax2 = Axis(fig2[1, 1], xlabel = "Time", ylabel = "Nº cells", title = "Cript mm concentration over time")
        lines!(ax2, (1:sizes), grid_values[1, :], label = "Cript 1")
        lines!(ax2, (1:sizes), grid_values[2, :], label = "Cript 2")
        lines!(ax2, (1:sizes), grid_values[3, :], label = "Cript 3")
        lines!(ax2, (1:sizes), grid_values[4, :], label = "Cript 4")
        lines!(ax2, (1:sizes), grid_values[5, :], label = "Cript 5")
        lines!(ax2, (1:sizes), grid_values[6, :], label = "Cript 6")
        lines!(ax2, (1:sizes), grid_values[7, :], label = "Cript 7")
        lines!(ax2, (1:sizes), grid_values[8, :], label = "Cript 8")
        fig2[1,2] = Legend(fig2, ax2, "Cripts")

        save("Plots/Cripts/Test_sims/concentration_time_n$(n)_lc$(depths[i]).png", fig2)
    end
end

Running simulation with parameters: 100 and 0.5


DimensionMismatch: DimensionMismatch: arrays could not be broadcast to a common size; got a dimension with lengths 1000 and 100000